# Classification demo

Anton Antonov  
September 2026

---

## Introduction

This notebook has a fully worked Machine Learning (ML) classification example using the Raku package "H2O::Client".

---

## Setup

In [83]:
use H2O::Client;

use Data::ExampleDatasets;
use Data::Generators;
use Data::Reshapers;
use Data::Summarizers;
use Data::TypeSystem;

use ML::ROCFunctions;
use Statistics::Distributions;

use JavaScript::D3;

In [84]:
#% javascript
require.config({
  paths: {
    d3: "https://d3js.org/d3.v7.min",
    d3_3d: "https://unpkg.com/d3-3d@2.0.2/build/d3-3d"
  }
});

require(["d3", "d3_3d"], function(d3, d3_3d) {
  window.d3 = d3;
  window.d33d = d3_3d || window.d33d || {};
  console.log("d3:", d3.version, "d3_3d:", Object.keys(window.d33d));
});

In [ ]:
#% js
js-d3-list-line-plot(100.rand xx 30, background => 'none')

----

## H2O client

It is assumed that a running H2O cluster is accessible via "http://localhost:54321":

In [86]:
my $h2o = H2O::Client('http://localhost:54321')

H2O::Client.new(base-url => "http://localhost:54321", timezone => -14400, transport => H2O::Client::Transport.new(base-url => "http://localhost:54321", timeout => 10))

---

## Data

In [87]:
my @dsExample = example-dataset('Stat2Data::Titanic');
#@dsExample .= map({ $_<Survived> = $_<Survived> ?? 'yes' !! 'no'; $_ });
#@dsExample .= map({ $_<Age> = (10 * round($_<Age> / 10)).Str; $_ });

sink records-summary(@dsExample);


+--------------------+-----------------+---------------------+-------------------------------------+--------------------+------------+---------------+
| Survived           | rownames        | Age                 | Name                                | SexCode            | PClass     | Sex           |
+--------------------+-----------------+---------------------+-------------------------------------+--------------------+------------+---------------+
| Min    => 0        | Min    => 1     | Min    => 0         | Kelly, Mr James             => 2    | Min    => 0        | 3rd => 711 | male   => 851 |
| 1st-Qu => 0        | 1st-Qu => 328.5 | 1st-Qu => 0         | Connolly, Miss Kate         => 2    | 1st-Qu => 0        | 1st => 322 | female => 462 |
| Mean   => 0.342727 | Mean   => 657   | Mean   => 17.502574 | Carlsson, Mr Frans Olof     => 2    | Mean   => 0.351866 | 2nd => 279 |               |
| Median => 0        | Median => 657   | Median => 18        | Hiltunen, Miss Marta        => 

###  Make training and testing datasets

In [88]:
my (@training-indexes, @testing-indexes);
with take-drop((^@dsExample.elems).pick(*), floor(@dsExample.elems * 0.75) ) {
        @training-indexes = $_.head;
        @testing-indexes = $_.tail;
}

my @field-names = <Age Sex PClass Survived>;

my @dsTraining = select-columns(@dsExample[@training-indexes], @field-names);
my @dsTesting = select-columns(@dsExample[@testing-indexes], @field-names);

say dimensions(@dsTraining);
say dimensions(@dsTesting);

(984 4)
(329 4)


### Summaries

Summary of the training table:

In [89]:
sink records-summary(@dsTraining, :@field-names);

+---------------------+---------------+------------+--------------------+
| Age                 | Sex           | PClass     | Survived           |
+---------------------+---------------+------------+--------------------+
| Min    => 0         | male   => 637 | 3rd => 526 | Min    => 0        |
| 1st-Qu => 0         | female => 347 | 1st => 241 | 1st-Qu => 0        |
| Mean   => 17.430437 |               | 2nd => 217 | Mean   => 0.345528 |
| Median => 18        |               |            | Median => 0        |
| 3rd-Qu => 30        |               |            | 3rd-Qu => 1        |
| Max    => 71        |               |            | Max    => 1        |
+---------------------+---------------+------------+--------------------+


Summary of the testing table:

In [90]:
sink records-summary(@dsTesting, :@field-names);

+---------------------+---------------+------------+--------------------+
| Age                 | Sex           | PClass     | Survived           |
+---------------------+---------------+------------+--------------------+
| Min    => 0         | male   => 214 | 3rd => 185 | Min    => 0        |
| 1st-Qu => 0         | female => 115 | 1st => 81  | 1st-Qu => 0        |
| Mean   => 17.718328 |               | 2nd => 62  | Mean   => 0.334347 |
| Median => 18        |               | *   => 1   | Median => 0        |
| 3rd-Qu => 30.5      |               |            | 3rd-Qu => 1        |
| Max    => 70        |               |            | Max    => 1        |
+---------------------+---------------+------------+--------------------+


### Upload, parse, and wait for both remote frames.

In [91]:
my $training-frame = $h2o.upload(
    @dsTraining,
    destination-frame => 'titanic-training.hex',
    column-names => @field-names,
    column-types => <Enum Enum Enum Enum>
).wait.result;

H2O::Frame<titanic-training.hex>[984 × 4]

In [92]:
my $testing-frame = $h2o.upload(
    @dsTesting,
    destination-frame => 'titanic-testing.hex',
    column-names => @field-names,
    column-types => <Enum Enum Enum Enum>
).wait.result;

H2O::Frame<titanic-testing.hex>[329 × 4]

In [93]:
say "Training frame: {$training-frame.gist}";
say "Testing frame: {$testing-frame.gist}";

Training frame: H2O::Frame<titanic-training.hex>[984 × 4]
Testing frame: H2O::Frame<titanic-testing.hex>[329 × 4]


----

## Jobs (so far)

In [94]:
#% html
my @dsJobs = $h2o.jobs('summary');
#say to-pretty-table(@dsJobs);
@dsJobs ==> to-html(field-names => @dsJobs.head.keys.sort.List)

description,destination,id,msec,progress,status
Parse,titanic-training.hex,$0301c0a801a732d4ffffffff$_b3eeabdf9853fa58ae89b575b9084630,67,1,DONE
Parse,titanic-testing.hex,$0301c0a801a732d4ffffffff$_b2d241cda3c64421b24d570130814ea9,7,1,DONE


---

## Model building and prediction

In [95]:
my %model-props =
        response_column => @field-names.tail,
        training_frame => "titanic-training.hex"
        ;

my $model-job = $h2o.model-build('drf', %model-props).wait;

H2O::Job<$0301c0a801a732d4ffffffff$_9b809ef5bf4bc981720d2bd22a3e4b96>[DONE 100%]

Show the H2O cluster models:

In [96]:
#% html
my @dsModels = $h2o.models('summary');

@dsModels ==> to-html()

mojo,response-column,pojo,id,algorithm,algo
True,Survived,True,DRF_model_HTTP_1788884176105_1,Distributed Random Forest,drf


Pick the last model from the list above:

In [97]:
my $model-id = $model-job.destination-id // @dsModels.tail<id>;
say "Using : {(:$model-id)}";

Using : model-id	DRF_model_HTTP_1788884176105_1


Model predictions:

In [98]:
my $predictions = $h2o.model-predict($model-id, $testing-frame.id, 'titanic-predictions.hex');

H2O::Frame<titanic-predictions.hex>[329 × 3]

Preview predictions result:

In [99]:
#% html
$predictions.preview
==> to-html()

p1,p0,predict
0.12263804435729986,0.8773619556427001,0
0.5380489044450223,0.46195109555497765,1
0.13202703237533564,0.8679729676246644,0
0.35863922124728564,0.6413607787527144,0
0.11551782727241511,0.8844821727275849,0
0.37446370601654055,0.6255362939834594,0
0.9449181831814348,0.05508181681856513,1
0.36572211861610415,0.6342778813838958,0
0.8060931839980185,0.1939068160019815,1
0.13202703237533564,0.8679729676246644,0


---

## Classifier measures

Here is a list of decision thresholds:

In [100]:
my @thresholds = 0.01, 0.02 ... 1;
@thresholds.elems

100

For each threshold calculate the prediction and make corresponding Receiver Operating Characteristic (ROC) record:

In [101]:
my @actual = $h2o.frame($testing-frame.id).Array.map(*<Survived>)».Int;
my @dsPredictions = $predictions.Array;
my @rocs = do for @thresholds -> $th {
    my @predicted = @dsPredictions.map({ $_<p1> ≥ $th ?? 1 !! 0 });
    to-roc-hash(1, 0, @actual, @predicted)
}

deduce-type(@rocs)

Vector(Assoc(Atom((Str)), Atom((Int)), 4), 100)

In [102]:
sink records-summary(@rocs)

+-----------------+-----------------+-----------------+------------------+
| FalsePositive   | FalseNegative   | TruePositive    | TrueNegative     |
+-----------------+-----------------+-----------------+------------------+
| Min    => 0     | Min    => 0     | Min    => 0     | Min    => 0      |
| 1st-Qu => 3.5   | 1st-Qu => 26    | 1st-Qu => 46    | 1st-Qu => 144    |
| Mean   => 49.78 | Mean   => 51.26 | Mean   => 58.74 | Mean   => 169.22 |
| Median => 11    | Median => 56    | Median => 54    | Median => 208    |
| 3rd-Qu => 75    | 3rd-Qu => 64    | 3rd-Qu => 84    | 3rd-Qu => 215.5  |
| Max    => 219   | Max    => 110   | Max    => 110   | Max    => 219    |
+-----------------+-----------------+-----------------+------------------+


Calculate ROC functions over the obtained ROC records:

In [103]:
my @funcs = (&PPV, &FPR, &TPR, &ACC, &MCC);
my @rocRes = @rocs.map( -> $r { @funcs.map({ $_.name => $_($r) }).Hash });
say to-pretty-table(@rocRes.pick(6));

+----------+----------+----------+----------+----------+
|   MCC    |   ACC    |   TPR    |   FPR    |   PPV    |
+----------+----------+----------+----------+----------+
| 0.502460 | 0.799392 | 0.490909 | 0.045662 | 0.843750 |
| 0.357221 | 0.586626 | 0.881818 | 0.561644 | 0.440909 |
| 0.371607 | 0.607903 | 0.863636 | 0.520548 | 0.454545 |
| 0.483475 | 0.787234 | 0.527273 | 0.082192 | 0.763158 |
| 0.500626 | 0.799392 | 0.472727 | 0.036530 | 0.866667 |
| 0.503216 | 0.793313 | 0.563636 | 0.091324 | 0.756098 |
+----------+----------+----------+----------+----------+


Here is the Area Under ROC (AUROC) value:

In [104]:
&AUROC(@rocs)

0.798111

Here is ROC-plot of FPR vs TPR:

In [ ]:
#% js
my %opts =  
    background => 'none',
    plot-label => 'ROC for Survived',
    plot-label-color => 'Gray',
    x-label => 'False Positive Rate (FPR)',
    y-label => 'True Positive Rate (TPR)',
    :450width,
    :450height,
    :grid-lines,
    :3stroke-width;

js-d3-list-line-plot(
    @rocRes.map({ <x y> Z=> $_<FPR TPR>})».Hash, |%opts
)